# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](https://github.com/mahmoodasayeedi/lab-neural-networks/blob/master/your-code/tttboard.jpg?raw=1)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

In [13]:
!git clone https://github.com/mahmoodasayeedi/lab-neural-networks.git

fatal: destination path 'lab-neural-networks' already exists and is not an empty directory.


Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [14]:
#Load the dataset
import pandas as pd

df = pd.read_csv('/content/lab-neural-networks/your-code/tic-tac-toe.csv')

df.head()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [15]:
#EDA
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB


In [16]:
# Check for missing values to determine reliability
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
TL       0
TM       0
TR       0
ML       0
MM       0
MR       0
BL       0
BM       0
BR       0
class    0
dtype: int64


In [17]:
#summarize
df.describe()

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
count,958,958,958,958,958,958,958,958,958,958
unique,3,3,3,3,3,3,3,3,3,2
top,x,x,x,x,x,x,x,x,x,True
freq,418,378,418,378,458,378,418,378,418,626


In [18]:
#1. Convert the categorical values to numeric in all columns.
from sklearn.preprocessing import LabelEncoder

# Initialize the label encoder
le = LabelEncoder()

# Loop through all columns and transform categorical data to numeric labels
for col in df.columns:
    df[col] = le.fit_transform(df[col])

# View data after transformation
print(df.head())



   TL  TM  TR  ML  MM  MR  BL  BM  BR  class
0   2   2   2   2   1   1   2   1   1      1
1   2   2   2   2   1   1   1   2   1      1
2   2   2   2   2   1   1   1   1   2      1
3   2   2   2   2   1   1   1   0   0      1
4   2   2   2   2   1   1   0   1   0      1


In [19]:
# Separate features (X) and label/target (y)
X = df.iloc[:, :-1]  # Selection of all columns except the last one
y = df.iloc[:, -1]   # Selection of only the last column

print(f"Inputs shape: {X.shape}, Output shape: {y.shape}")

Inputs shape: (958, 9), Output shape: (958,)


In [20]:
from sklearn.preprocessing import MinMaxScaler

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Fit to the input data and transform it
X_normalized = scaler.fit_transform(X)

# Convert back to a DataFrame for clean visualization (optional)
X_normalized_df = pd.DataFrame(X_normalized, columns=X.columns)
print(X_normalized_df.head())


    TL   TM   TR   ML   MM   MR   BL   BM   BR
0  1.0  1.0  1.0  1.0  0.5  0.5  1.0  0.5  0.5
1  1.0  1.0  1.0  1.0  0.5  0.5  0.5  1.0  0.5
2  1.0  1.0  1.0  1.0  0.5  0.5  0.5  0.5  1.0
3  1.0  1.0  1.0  1.0  0.5  0.5  0.5  0.0  0.0
4  1.0  1.0  1.0  1.0  0.5  0.5  0.0  0.5  0.0


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [21]:
# 1. Split the training and test data.
from sklearn.model_selection import train_test_split

# Split into 80% training and 20% test data
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.2, random_state=42
)


In [26]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Initialize the Sequential model
model = Sequential()

# Input layer + first middle (hidden) layer with ReLU
model.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))

# Additional middle (hidden) layer with ReLU
model.add(Dense(16, activation='relu'))

# Output layer with Softmax (assumes 2 unique class labels for the outcome)
model.add(Dense(2, activation='softmax'))



In [28]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [29]:
# Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1
)


Epoch 1/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 5s 125ms/step - accuracy: 0.6502 - loss: 0.6505 - val_accuracy: 0.7143 - val_loss: 0.6093
Epoch 2/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6517 - loss: 0.6357 - val_accuracy: 0.7143 - val_loss: 0.6006
Epoch 3/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6546 - loss: 0.6274 - val_accuracy: 0.7143 - val_loss: 0.5940
Epoch 4/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6589 - loss: 0.6204 - val_accuracy: 0.7143 - val_loss: 0.5870
Epoch 5/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6618 - loss: 0.6134 - val_accuracy: 0.7143 - val_loss: 0.5810
Epoch 6/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6618 - loss: 0.6086 - val_accuracy: 0.7273 - val_loss: 0.5780
Epoch 7/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6647 - loss: 0.6039 - val_accuracy: 0.7273 - val_loss: 0.5745
Epoch 8/50
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6880 - loss: 0.5996 - val_accuracy: 0.7403 - val_los

In [30]:
## Evaluate accuracy on test data
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {test_accuracy * 100:.2f}%")


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7500 - loss: 0.4705 

Test Accuracy: 75.00%


In [32]:
# Save the model using the recommended native Keras format
model.save('tic-tac-toe.keras')
print("Model successfully saved as tic-tac-toe.keras")



Model successfully saved as tic-tac-toe.keras


## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [23]:
import numpy as np
import pandas as pd
import tensorflow as tf

# 1. Load your saved model
model = tf.keras.models.load_model('tic-tac-toe.keras')

# 2. Select 5 random indices from the test dataset
# (Assumes X_test and y_test are numpy arrays or dataframes from your split step)
np.random.seed(42)  # For reproducible random choices
random_indices = np.random.choice(len(X_test), size=5, replace=False)

# 3. Extract the random samples and their true labels
# If X_test and y_test are DataFrames/Series, use .iloc. If numpy arrays, use slicing.
if isinstance(X_test, pd.DataFrame):
    samples = X_test.iloc[random_indices]
    true_labels = y_test.iloc[random_indices].values
else:
    samples = X_test[random_indices]
    true_labels = y_test[random_indices]

# 4. Make predictions
# The model outputs probabilities for each class (e.g., [0.15, 0.85]) due to softmax
raw_predictions = model.predict(samples)

# Use argmax to pick the class index with the highest probability (0 or 1)
predicted_classes = np.argmax(raw_predictions, axis=1)

# 5. Display and compare the results
print("\n--- Prediction Results ---")
for i in range(len(random_indices)):
    print(f"Sample {i+1}:")
    print(f"  Predicted Class: {predicted_classes[i]}")
    print(f"  Actual Class:    {true_labels[i]}")

    # Check if the model got it right
    if predicted_classes[i] == true_labels[i]:
        print("  Result:          ✅ Correct!")
    else:
        print("  Result:          ❌ Incorrect")
    print("-" * 25)


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [24]:
# your code here

**Which approach(es) did you find helpful to improve your model performance?**

In [25]:
# your answer here